# Hybrid Search and Reranking

Build BM25 keyword retrieval and combine its ranking with vector
retrieval using Reciprocal Rank Fusion.

Both retrieval methods rank the same chunk IDs.

Prerequisites:
- Saved chunk records from notebook 02.
- Populated Chroma collection from notebook 03.

Run cells in order with the project environment. CUDA is used for model inference.

## 1. Load chunk records and libraries

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import chromadb

from rank_bm25 import BM25Okapi

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

chunks_path = project_root / "data" / "processed" / "job_chunks.jsonl"

chunks_df = pd.read_json(
    chunks_path,
    lines=True,
    convert_dates=False
)

assert chunks_df["chunk_id"].is_unique

print("Loaded chunks:", len(chunks_df))

Loaded chunks: 3618


## 2. Tokenize keywords

Apply the same normalization to documents and queries. CJK characters are separated as a basic fallback, not linguistic word segmentation. No translation, stemming, or stopword removal is performed.

In [2]:
def tokenize_keywords(text):
    text = text.casefold()

    text = re.sub(
        r"([\u4e00-\u9fff\u3040-\u30ff])",
        r" \1 ",
        text
    )

    return re.findall(r"c\+\+|c#|[^\W_]+", text)

sample_text = "Python APIs, MySQL, C++ and C# 软件开发"

print(tokenize_keywords(sample_text))

['python', 'apis', 'mysql', 'c++', 'and', 'c#', '软', '件', '开', '发']


## 3. Build the BM25 corpus

Index job metadata, including full locations, and description chunks. Keyword matching does not enforce a strict location constraint.

In [3]:
keyword_columns = [
    "Job Title",
    "Company Name",
    "Job Category",
    "Job Level",
    "Job Location",
    "Tags",
    "chunk_text"
]

keyword_texts = (
    chunks_df[keyword_columns]
    .fillna("")
    .agg("\n".join, axis=1)
    .tolist()
)

tokenized_corpus = [
    tokenize_keywords(text)
    for text in keyword_texts
]

assert all(tokenized_corpus)

bm25 = BM25Okapi(tokenized_corpus)

print("Indexed chunks:", bm25.corpus_size)
print("First document's first 20 terms:", tokenized_corpus[0][:20])

Indexed chunks: 3618
First document's first 20 terms: ['dir', 'equities', 'quant', 'merrill', 'data', 'and', 'analytics', 'mid', 'level', 'new', 'york', 'ny', 'job', 'description', 'at', 'bank', 'of', 'america', 'we', 'are']


## 4. Inspect the BM25 baseline

In [4]:
query = "Python backend developer working with APIs and databases"

query_tokens = tokenize_keywords(query)
bm25_scores = bm25.get_scores(query_tokens)

ranked_positions = np.argsort(-bm25_scores, kind="stable")
positive_positions = ranked_positions[
    bm25_scores[ranked_positions] > 0
]
top_positions = positive_positions[:5]

bm25_results = chunks_df.iloc[top_positions][
    ["chunk_id", "ID", "Job Title", "Company Name"]
].copy()

bm25_results.insert(0, "rank", range(1, len(bm25_results) + 1))
bm25_results["bm25_score"] = bm25_scores[top_positions]

print("Query terms:", query_tokens)
display(bm25_results)

Query terms: ['python', 'backend', 'developer', 'working', 'with', 'apis', 'and', 'databases']


,rank,chunk_id,ID,Job Title,Company Name,bm25_score
866,1,LF0247_chunk_001,LF0247,Backend Software Engineer - Global E-commerce ...,TikTok,22.605108
890,2,LF0255_chunk_000,LF0255,.NET Developer with Angular,EPAM Systems,21.831936
819,3,LF0232_chunk_001,LF0232,"Senior Backend Software Engineer, Content E-co...",TikTok,19.847482
1191,4,LF0343_chunk_002,LF0343,Senior Software Engineer - Master Data Management,Ahold Delhaize USA,19.454783
822,5,LF0233_chunk_001,LF0233,Principal Backend Software Engineer,Atlassian,18.612064


## 5. Reopen Chroma and verify matching chunk IDs

In [5]:
database_path = project_root / "data" / "indexes" / "chroma"

client = chromadb.PersistentClient(path=str(database_path))

collection = client.get_collection(
    name="job_chunks_e5_small_v1",
    embedding_function=None
)

stored_ids = set(collection.get(include=[])["ids"])
expected_ids = set(chunks_df["chunk_id"])

assert stored_ids == expected_ids

print("Matching chunk IDs verified:", len(stored_ids))

Matching chunk IDs verified: 3618


## 6. Retrieve vector candidates

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-small",
    device="cuda"
)

query_embedding = embedding_model.encode(
    ["query: " + query],
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False
)

CANDIDATE_COUNT = 20

vector_results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=CANDIDATE_COUNT,
    include=["documents", "metadatas", "distances"]
)

vector_ranked_ids = vector_results["ids"][0]

print("Vector candidates:", len(vector_ranked_ids))

W1028 13:54:42.229000 404 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/intfloat/multilingual-e5-small: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'intfloat/multilingual-e5-small'.


Vector candidates: 20


## 7. Fuse the rankings

Combine up to 20 candidates from each retriever using equal-weight Reciprocal Rank Fusion with k=60. These are initial settings, not optimized parameters. Candidate IDs are deduplicated at chunk level.

In [7]:
bm25_ranked_ids = chunks_df.iloc[
    positive_positions[:CANDIDATE_COUNT]
]["chunk_id"].tolist()

RRF_K = 60
fusion_scores = {}

for ranked_ids in [vector_ranked_ids, bm25_ranked_ids]:
    for rank, chunk_id in enumerate(ranked_ids, start=1):
        contribution = 1.0 / (RRF_K + rank)

        fusion_scores[chunk_id] = (
            fusion_scores.get(chunk_id, 0.0) + contribution
        )

fused_ids = sorted(
    fusion_scores,
    key=lambda chunk_id: (-fusion_scores[chunk_id], chunk_id)
)

hybrid_results = chunks_df.set_index("chunk_id").loc[
    fused_ids,
    ["ID", "Job Title", "Company Name"]
].copy()

hybrid_results.insert(0, "rank", range(1, len(hybrid_results) + 1))
hybrid_results["rrf_score"] = [
    fusion_scores[chunk_id] for chunk_id in fused_ids
]

print("Combined unique candidate chunks:", len(fused_ids))
display(hybrid_results.head(10))

Combined unique candidate chunks: 34


,rank,ID,Job Title,Company Name,rrf_score
chunk_id,,,,,
LF0247_chunk_001,1,LF0247,Backend Software Engineer - Global E-commerce ...,TikTok,0.032522
LF0255_chunk_000,2,LF0255,.NET Developer with Angular,EPAM Systems,0.029828
LF0264_chunk_002,3,LF0264,"Senior Software Engineer, Infrastructure (Plat...",Coinbase,0.028814
LF0265_chunk_000,4,LF0265,Senior Python Developer (with GenAI),EPAM Systems,0.027027
LF0321_chunk_001,5,LF0321,Software Engineer TS/SCI Poly,Leidos,0.026743
LF0309_chunk_001,6,LF0309,Software Engineer TS/SCI Poly,Leidos,0.026519
LF0117_chunk_002,7,LF0117,"Backend Software Engineer, TikTok Data Ecosyst...",TikTok,0.016393
LF0117_chunk_001,8,LF0117,"Backend Software Engineer, TikTok Data Ecosyst...",TikTok,0.015873
LF0232_chunk_001,9,LF0232,"Senior Backend Software Engineer, Content E-co...",TikTok,0.015873


## Cross-encoder reranking

Score the fused candidates with a multilingual cross-encoder, then sort by that
score. Whether this improves retrieval needs a labeled comparison.


### Load the multilingual reranker

In [8]:
from sentence_transformers import CrossEncoder

reranker_model_name = "BAAI/bge-reranker-v2-m3"

print("Loading reranker", flush=True)

reranker = CrossEncoder(
    reranker_model_name,
    device="cuda",
    max_length=512
)

print("Reranker loaded.")

Loading reranker


Could not reach the Hub (Cannot reach https://huggingface.co/api/models/BAAI/bge-reranker-v2-m3: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'BAAI/bge-reranker-v2-m3'.


Reranker loaded.


### Prepare pairs and check token limits

Remove the E5 passage prefix and measure combined query/document lengths using the reranker's tokenizer before scoring.

In [9]:
candidate_rows = chunks_df.set_index("chunk_id").loc[fused_ids].copy()

candidate_texts = (
    candidate_rows["embedding_text"]
    .str.removeprefix("passage: ")
    .tolist()
)

reranker_pairs = [
    (query, text)
    for text in candidate_texts
]

pair_token_counts = [
    len(reranker.tokenizer(
        query,
        text,
        add_special_tokens=True,
        truncation=False
    )["input_ids"])
    for text in candidate_texts
]

print("Candidate pairs:", len(reranker_pairs))
print("Longest pair:", max(pair_token_counts))
print("Pairs exceeding 512 tokens:", sum(
    count > 512 for count in pair_token_counts
))

Candidate pairs: 34
Longest pair: 435
Pairs exceeding 512 tokens: 0


### Score and reorder candidates

In [10]:
assert max(pair_token_counts) <= 512

print("Scoring candidate pairs", flush=True)

reranker_scores = reranker.predict(
    reranker_pairs,
    batch_size=4,
    show_progress_bar=False,
    convert_to_numpy=True
)

reranker_scores = np.asarray(reranker_scores).reshape(-1)

assert len(reranker_scores) == len(candidate_rows)
assert np.isfinite(reranker_scores).all()

reranked_results = candidate_rows.copy()
reranked_results["rrf_score"] = [
    fusion_scores[chunk_id]
    for chunk_id in reranked_results.index
]
reranked_results["reranker_score"] = reranker_scores

reranked_results = reranked_results.sort_values(
    "reranker_score",
    ascending=False,
    kind="stable"
)

display(
    reranked_results[
        ["ID", "Job Title", "Company Name", "rrf_score", "reranker_score"]
    ].head(10)
)

Scoring candidate pairs


,ID,Job Title,Company Name,rrf_score,reranker_score
chunk_id,,,,,
LF0265_chunk_000,LF0265,Senior Python Developer (with GenAI),EPAM Systems,0.027027,0.192541
LF0247_chunk_001,LF0247,Backend Software Engineer - Global E-commerce ...,TikTok,0.032522,0.172211
LF0237_chunk_000,LF0237,"Lead Software Engineer, Back End",Capital One,0.012987,0.125938
LF0217_chunk_002,LF0217,Site Reliability Engineer,MIO Partners Inc.,0.014085,0.089479
LF0192_chunk_001,LF0192,Senior Python Data Engineer,EPAM Systems,0.014286,0.087497
LF0192_chunk_000,LF0192,Senior Python Data Engineer,EPAM Systems,0.014706,0.084587
LF0264_chunk_002,LF0264,"Senior Software Engineer, Infrastructure (Plat...",Coinbase,0.028814,0.074182
LF0232_chunk_001,LF0232,"Senior Backend Software Engineer, Content E-co...",TikTok,0.015873,0.072991
LF0233_chunk_001,LF0233,Principal Backend Software Engineer,Atlassian,0.015385,0.062558


## 8. Present distinct jobs

Keep each job's best-scoring candidate chunk. This does not combine evidence across the full job description.

In [11]:
top_jobs = (
    reranked_results
    .drop_duplicates(subset="ID", keep="first")
    .head(5)
    .copy()
)

top_jobs.insert(0, "rank", range(1, len(top_jobs) + 1))

display(
    top_jobs[
        ["rank", "ID", "Job Title", "Company Name", "reranker_score"]
    ]
)

,rank,ID,Job Title,Company Name,reranker_score
chunk_id,,,,,
LF0265_chunk_000,1,LF0265,Senior Python Developer (with GenAI),EPAM Systems,0.192541
LF0247_chunk_001,2,LF0247,Backend Software Engineer - Global E-commerce ...,TikTok,0.172211
LF0237_chunk_000,3,LF0237,"Lead Software Engineer, Back End",Capital One,0.125938
LF0217_chunk_002,4,LF0217,Site Reliability Engineer,MIO Partners Inc.,0.089479
LF0192_chunk_001,5,LF0192,Senior Python Data Engineer,EPAM Systems,0.087497


## 9. Inspect the highest-ranked evidence

In [12]:
best_job = top_jobs.iloc[0]

print("Job ID:", best_job["ID"])
print("Title:", best_job["Job Title"])
print("Evidence chunk ID:", best_job.name)
print("\nRetrieved text:\n")
print(best_job["chunk_text"])

Job ID: LF0265
Title: Senior Python Developer (with GenAI)
Evidence chunk ID: LF0265_chunk_000

Retrieved text:

We are seeking a
Senior Python Developer (with GenAI)
to join the project, which involves adoption of Artificial Intelligence solutions by our clients and focusing on generative AI.
In this role, you will be responsible for delivering multiple Proof of Concepts (POCs) and solving challenges using generative AI. Our approach is fluid and dynamic. This is a role that offers the freedom to innovate and evolve.
#LI-DNI
Responsibilities
Design and integrate Generative AI solutions with existing systems or create new applications, ensuring the adoption of the latest advancements in Generative AI
Act as the main point of contact for clients, responsible for gathering requirements and providing status updates
Communicate project progression to stakeholders, and present suitable architecture solutions based on business requests
Collaborate with clients throughout the project timeline

## Reranking observations

For the Python/backend/API/database query, LF0265 ranks first and LF0247 second.
LF0265 explicitly requires Python, but this chunk does not establish API or
database work. LF0247 supports backend development, Python as one accepted
language, and databases; API work is not explicit there either.

Both are partial matches to the full question. Scores order candidates; they do
not prove that every requested skill is present. These observations use retrieved
chunks rather than full descriptions. See docs/evaluation.md for the later comparison.
